# 1. Introduction and Business Idea

##1.1 Created by (Team 1)

Anna Calkins (100578235)

Clark Enge (100578112)

Eline Varhaugvik (100576807)

Maya Jha (100578695)

Martyna Przeradzka (100576084)

Sofia Ferreira Hidalgo de Morillo (100524514)

## 1.2 Business idea
This project focuses exclusively on Airbnb listings in Los Angeles, with an emphasis on understanding how property characteristics and neighborhood context jointly influence short-term rental revenue. The central research question is:

***“Which combinations of property attributes, specifically bedroom and bathroom configurations, yield the highest Airbnb revenue across different neighborhoods in Los Angeles?”***


---


The analysis will be conducted using entire-home/entire-apartment listings only, ensuring comparability among properties intended for full private occupancy. To examine investment attractiveness at a granular level, the study will segment the dataset along the following dimensions:

- Neighborhoods within Los Angeles
- Bedroom–bathroom combinations (property configuration)
- Entire-home listings exclusively
- Average neighborhood revenue compared with predicted revenue for specific property types


---


For the purposes of this study, we have chosen to focus on five neighbourhoods within Los Angeles that are known for hosting major events. These neighbourhoods were selected based on the presence of large, well-established event venues. The selected neighbourhoods and their corresponding venues are as follows:

- Hollywood Hills → Hollywood Bowl
- Inglewood → Kia Forum
- Downtown LA → Crypto.com Arena
- Exposition Park → Los Angeles Memorial Coliseum
- Pasadena → Rose Bowl Stadium


The business objective of this research is to provide property owners, real-estate investors, and rental managers with data-driven guidance on whether investing in or renting out a property in a specific Los Angeles neighborhood is likely to be financially advantageous.


# 2. Data Loading, Organizing, Cleaning

## 2.1 Importing Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


## 2.2 Data Loading

In [ ]:
#Data Loading
from google.colab import drive
drive.mount('/content/drive')

#Important we are using csv.gz!!!!!
listings_df = pd.read_csv('/content/drive/MyDrive/BigDataFinalProject/LosAngeles/listings.csv.gz')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2.3 Data Cleaning

In [ ]:
#Create a DataFrame  containg the relevant variables
listings_clean_df = listings_df[['id',
                                 'bathrooms',
                                 'bedrooms',
                                 'neighbourhood_cleansed',
                                 'neighbourhood_group_cleansed',
                                 'property_type',
                                 'room_type',
                                 'accommodates',
                                 'price',
                                 'estimated_occupancy_l365d',
                                 'estimated_revenue_l365d'
                                 ]]

private_df = listings_clean_df[listings_clean_df['room_type'] == 'Entire home/apt'].copy()

#___________________________________________________________________________
# Clean price (remove "$" and commas if present)
private_df["price"] = private_df["price"].astype(str).str.replace("$", "", regex=False).str.replace(",", "")

# Convert to numeric
private_df["price"] = pd.to_numeric(private_df["price"], errors="coerce")
#___________________________________________________________________________

#Removing rows where Bathrooms, Bedrooms, and or Price is NA or 0
private_df = private_df.query("bathrooms == bathrooms and bathrooms != 0 and bedrooms == bedrooms and bedrooms != 0 and price == price and price != 0")

#Changes name of estimated revenue category to revenue
private_df.rename(columns={'estimated_revenue_l365d' : 'revenue'}, inplace=True)


#___________________________________________________________________________
#Seperate neighborhood df
inglewood_df = private_df[private_df['neighbourhood_cleansed'] == 'Inglewood'].copy()
hollywood_hills_df = private_df[private_df['neighbourhood_cleansed'] == 'Hollywood Hills'].copy()
pasadena_df = private_df[private_df['neighbourhood_cleansed'] == 'Pasadena'].copy()
downtown_df = private_df[private_df['neighbourhood_cleansed'] == 'Downtown'].copy()
exposition_park_df = private_df[private_df['neighbourhood_cleansed'] == 'Exposition Park'].copy()

#Combines all the dataframes into one
combined_df = pd.concat([
    inglewood_df,
    hollywood_hills_df,
    pasadena_df,
    downtown_df,
    exposition_park_df
], axis=0).copy()


#___________________________________________________________________________
#This creates a new dataframe, not a view, with the necessary columns
model_df = combined_df[['revenue', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed']].copy()

#This creates the dummy variables to represent the neighbourhood data
model_df = pd.get_dummies(model_df, columns=['neighbourhood_cleansed'], drop_first=False)


To clean and organize the data, there were a few key steps we took. First, we removed columns that were irrelevant to our investigation. These included all of the review-related columns, as our main measurement is revenue, and others, like minimum and maximum night related columns.

Then, we removed any entries where the bedrooms or bathrooms or estimated revenue over the last 365 days was not available or zero, since these were the crucial input (number of bedrooms and bathrooms) and output (revenue) to our analysis.

We broke down the dataframe with all listings to five different dataframes for the five neighborhoods that were of interest to us, as they were closest to main venues in LA. We combined these five dataframes into one to have all listings from these five neighborhoods congregated.

Finally, we one-hot encoded the neighbourhood_cleansed column, creating five dummy variables (one per neighborhood). Each listing is assigned a 1 in the dummy column corresponding to its neighborhood and 0 in all others.